# C1-ml-fundamentals — Unit Review

Use this notebook after finishing the three sessions and (ideally) the
practice sets: skim the summaries, take the self-quiz cold, then follow the
redo pointers for anything that felt shaky.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

## Concept summary

| Concept | One-line summary | Session |
|---|---|---|
| supervised vs. unsupervised | Supervised = the answer is a label column in the data; unsupervised = no labels, find structure. | 1 |
| clustering | Group unlabeled items so similar ones share a group; delivers membership, never meaning or names. | 1 |
| train/test split | Seeded shuffle, same permutation for inputs and labels, slice; the test set is touched exactly once, at the end. | 2 |
| overfitting | Much better on training than on new data — the rule latched onto specifics; the train-test gap is the tell. | 2 |
| `bias-variance-intuition` | Rigid rules err systematically (poor everywhere); flexible rules soak up quirks (great train, poor test); good rules sit between. | 2 |
| accuracy / precision / recall | Fraction right overall; fraction of positive *calls* that were right; fraction of *real* positives found. | 3 |
| F1 | Harmonic mean of precision and recall — stays low unless both are decent. | 3 |
| macro-F1 | Per-class one-vs-rest F1, averaged with equal class weights; rare classes count fully. | 3 |
| class imbalance | When one class dominates, accuracy flatters do-nothing rules; compare to the majority baseline, then use recall/F1. | 3 |

## Formula and idiom sheet

**Metric formulas** (positive class = label `1`):

- accuracy = (TP + TN) / (TP + TN + FP + FN)
- precision = TP / (TP + FP)  •  recall = TP / (TP + FN)
- F1 = 2 · precision · recall / (precision + recall); convention: F1 = 0
  when no true positives are found
- per-class (table rows = actual): precision_k = C[k,k] / column-k sum,
  recall_k = C[k,k] / row-k sum; macro-F1 = mean of per-class F1
- majority baseline accuracy = larger class fraction of `y_true`

**NumPy idioms** (all from F1-scientific-python):

```python
TP = np.sum((y_true == 1) & (y_pred == 1))        # bucket counts via masks
order = np.random.default_rng(SEED).permutation(n) # seeded shuffle
X_tr, y_tr = X[order][:k], y[order][:k]            # same order, both arrays
diag = np.diag(C); C.sum(axis=0); C.sum(axis=1)    # table -> per-class sums
```

## Self-quiz (13 items)

Work every item without looking anything up; answers are collapsed at the
end.

1. Classify: predict delivery time for new orders from past deliveries,
   each recorded with its actual delivery time.
2. True or false: what makes a task supervised is that the answers exist as
   a label column *in the dataset*, not merely in an expert's head.
3. What does clustering output, and what must a human still supply?
4. Why must the same permutation reorder both `X` and `y` in a split?
5. After a seeded 80/20 shuffle-split of 40 labeled examples, what are the
   shapes of the four arrays?
6. A rule scores 99% train / 68% test. Name the phenomenon and the number
   pair that reveals it.
7. Where does the memorizer sit on the rigid–flexible spectrum, and what is
   the symptom of each end?
8. TP = 10, FP = 5, FN = 2, TN = 33. Compute accuracy, precision, recall.
9. Continuing item 8: compute F1 as a fraction in lowest terms.
10. For the 2-class table `[[8, 2], [3, 7]]` (rows = actual), compute both
    per-class F1 scores and the macro-F1 (decimals fine).
11. A dataset is 1% positive. What accuracy does "always negative" score,
    what is its recall, and which metric family exposes it?
12. A screening program's misses are far costlier than its false alarms.
    Which metric should drive its evaluation, and why?
13. Spot the leakage: a cutoff is chosen using candidate midpoints computed
    from train **and** test values, then "evaluated" on the test set.

In [ ]:
# Optional: verify your item 8-10 arithmetic.
TP, FP, FN, TN = 10, 5, 2, 33
acc = (TP + TN) / 50
prec = TP / 15
rec = TP / 12
f1 = 2 * prec * rec / (prec + rec)
print(f"Q8: accuracy={acc}  precision={prec:.4f}  recall={rec:.4f}")
print(f"Q9: F1={f1:.4f}  (20/27 = {20 / 27:.4f})")

C2x2 = np.array([[8, 2], [3, 7]])
d = np.diag(C2x2).astype(float)
p = d / C2x2.sum(axis=0)
r = d / C2x2.sum(axis=1)
f = 2 * p * r / (p + r)
print(f"Q10: per-class F1 = {np.round(f, 4)}  macro-F1 = {f.mean():.4f}")

## Self-quiz answers

<details><summary><b>Show all answers</b></summary>

1. **Supervised** — each past delivery carries the answer (its actual time).
2. **True.** Expert knowledge becomes supervision only once it is recorded
   as labels.
3. The groups themselves (which items belong together). A human supplies
   the interpretation and the names.
4. Each label belongs to one specific input; independent shuffles pair
   inputs with other examples' labels and the dataset becomes scrambled
   nonsense.
5. `(32,)`, `(32,)`, `(8,)`, `(8,)`.
6. **Overfitting**; the train-test gap (99% vs 68%).
7. The extreme flexible end. Flexible-end symptom: great train, poor test
   (the gap). Rigid-end symptom: train and test both poor and close
   together.
8. accuracy = 43/50 = 0.86; precision = 10/15 = 2/3; recall = 10/12 = 5/6.
9. F1 = 2 · (2/3)(5/6) / (2/3 + 5/6) = (10/9)/(3/2) = **20/27** (gcd(20,
   27) = 1).
10. Class 0: precision 8/11, recall 8/10 → F1 = 16/21 ≈ 0.762. Class 1:
    precision 7/9, recall 7/10 → F1 = 14/19 ≈ 0.737. Macro-F1 ≈ **0.749**.
11. Accuracy 0.99; recall 0; the metrics that respect the rare class —
    recall and F1 (with precision undefined for a rule that never flags).
12. **Recall** — it counts the fraction of real cases caught, and a miss is
    the costly error; false alarms cost only follow-ups (precision tracks
    that burden).
13. The candidate list depends on test values, so the test set influenced
    rule-building before the final measurement — the reported score is
    inflated; candidates must come from training data only.

</details>

## What to redo, per weak spot

| Shaky on… | Redo |
|---|---|
| task types / clustering | p01, p15, p17, p21, p22 |
| train/test splitting | p02, p16, p20 (a) |
| overfitting and the gap | p03, p19, p23, p07 |
| flexible vs. rigid reasoning | p03 (task 3), p07, p19 (c) |
| confusion counts and the metric formulas | p04, p09, p10 |
| F1 and its behaviour | p05, p09, p12 (c), p18 |
| macro-F1 | p06, p11, p13 |
| class imbalance judgment | p05, p08, p13 (c), p18, p21 |
| the full workflow under exam constraints | p14, p20 |

If a whole session felt thin, re-run its notebook top to bottom and attempt
every checkpoint again before returning to the practice sets.